# Step 4 Extension: Annotate Remaining 250 + Human Labelling File

1. Annotate rows 251-500 with condition B (few-shot Ministral)
2. Generate a clean human labelling file for you and Robin

In [11]:
from pathlib import Path
import pandas as pd
import numpy as np
import requests
import time
import os
import krippendorff
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score, classification_report,
    f1_score, cohen_kappa_score, confusion_matrix
)
import importlib
import annotation_setup
importlib.reload(annotation_setup)
from annotation_setup import VALID_LABELS, instructions, reminder, parse_label, parse_explanation

load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "ministral-3-14b"

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Model: {MODEL}")

Model: ministral-3-14b


In [12]:
FEW_SHOT_EXAMPLES = [
    # 1. Clear NO_CRIME_FRAME, integration context, no security link
    {
        "text": "Die Lage in der Unterkunft hat sich nach Angaben der Sozialarbeiter entspannt. Viele [Gruppe 1] haetten inzwischen Deutschkurse begonnen und bemuehten sich aktiv um eine Arbeitsstelle. Die Caritas lobte das Engagement der Freiwilligen, die regelmaessig Unterstuetzung anbieten.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Der Absatz beschreibt Integration und ehrenamtliche Hilfe ohne jeden Bezug zu Kriminalitaet, Sicherheit oder Praevention."
    },
    # 2. Clear CRIME_FRAME_NEG, direct perpetrator
    {
        "text": "Ein 34-jaehriger [Gruppe 1] wurde am Freitagabend am Hauptbahnhof festgenommen. Er soll in den vergangenen Wochen mehrfach Passanten mit einem Messer bedroht und Handtaschen entrissen haben. Die Polizei hat ihn als Serientaeter eingestuft.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Ein Gruppenmitglied wird explizit als Taeter schwerer Straftaten dargestellt und direkt mit Kriminalitaet verknuepft."
    },
    # 3. CRIME_FRAME_NEG, institutional security framing (fixes missed cases)
    {
        "text": "Die Innenministerkonferenz hat beschlossen, die erkennungsdienstliche Erfassung aller einreisenden [Gruppe 1] zu verschaerfen. Innenminister Hoffmann erklaerte, man koenne sich keine Sicherheitsluecken leisten. Es muesse sichergestellt werden, dass keine Schleuser, Kriminellen oder Extremisten die Situation ausnutzten.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Auch ohne konkrete Straftat wird die Gruppe institutionell als potenzielle Sicherheitsgefahr gerahmt — die explizite Verknuepfung mit Kriminalitaet und Extremismus durch eine Regierungsstimme ist CRIME_FRAME_NEG."
    },
    # 4. NO_CRIME_FRAME, group is victim of violence
    {
        "text": "Auf dem Gelaende der Fluechtlingsunterkunft in Tempelhof wurden in der Nacht Fensterscheiben eingeworfen und Hakenkreuze gesprueht. Die Polizei ermittelt wegen schwerer Sachbeschaedigung und Volksverhetzung. Die [Gruppe 1] Bewohner wurden unverletzt, aber stark veraengstigt in eine Notunterkunft gebracht.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Die [Gruppe 1] sind Opfer eines rechtsextremen Angriffs — eine Gruppe als Opfer von Gewalt darzustellen ist kein Sicherheitsrahmen gegen diese Gruppe."
    },
    # location not mentioned and no Germany link = NO_CRIME_FRAME
    {
        "text": "Berichte zufolge wurden mehrere [Gruppe 1] bei Auseinandersetzungen verletzt. Wo genau die Vorfaelle stattfanden, blieb unklar. Ein Bezug zu Deutschland oder deutschen Sicherheitsbehoerden wurde nicht hergestellt.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Kein expliziter Bezug zu Deutschland oder zur deutschen inneren Sicherheit — ohne diesen Bezug gilt NO_CRIME_FRAME."
    },
    # 5. NO_CRIME_FRAME, international war reporting between states
    {
        "text": "Die [Gruppe 1] Streitkraefte haben nach Angaben des Verteidigungsministeriums in Kiew erneut mehrere Staedte im Osten beschossen. Bei den Angriffen kamen nach ukrainischen Angaben mindestens zwoelf Zivilisten ums Leben. Die NATO beriet ueber weitere Waffenlieferungen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Kriegsberichterstattung ueber einen Konflikt zwischen Staaten ohne Bezug zu Minderheiten in Deutschland erhaelt immer NO_CRIME_FRAME."
    },
    # Germany-link exception, foreign crime but explicit Germany connection = CRIME_FRAME_NEG
    {
        "text": "Der Attentaeter, der in Wien drei Menschen toetete, hatte mehrere Jahre in Deutschland gelebt und war deutschen Behoerden als Gefaehrder bekannt. Das Bundesinnenministerium prueft nun, ob Sicherheitsluecken im deutschen System vorlagen.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Obwohl die Tat in Oesterreich stattfand, wird ein expliziter Bezug zur deutschen Sicherheit hergestellt — der Taeter lebte in Deutschland und deutsche Behoerden sind beteiligt. Das ergibt CRIME_FRAME_NEG."
    },
    # 6. CRIME_FRAME_NEG, group legitimizes terrorism without being direct perpetrator
    {
        "text": "Sprecher der [Gruppe 1] bezeichneten den Anschlag als legitimen Widerstand und erklaerten, die Opfer seien selbst schuld. In sozialen Medien verbreiteten Anhaenger der Gruppe Videos, in denen der Terrorakt gefeiert wurde.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe legitimiert und feiert explizit einen Terroranschlag — das zaehlt als CRIME_FRAME_NEG auch ohne direkte Taeterschaft."
    },
    # 7. CRIME_FRAME_POS, successful security operation
    {
        "text": "Das Gemeinsame Terrorismusabwehrzentrum hat nach intensiver Zusammenarbeit von Polizei und Geheimdiensten einen islamistischen Anschlag in Muenchen vereitelt. Drei Verdaechtige wurden festgenommen. Innenministerin Weber sprach von einem grossen Erfolg der Sicherheitsbehoerden.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Die Sicherheitsbehoerden werden explizit als erfolgreiche Akteure gerahmt die eine Bedrohung abgewendet haben — Schutz und Sicherheitsgewinn stehen im Vordergrund."
    },
    # 8. CRIME_FRAME_POS, social factors explanation + differentiation (aligns with Robin's definition)
    {
        "text": "Experten erklaeren den Anstieg der Jugendkriminalitaet in Brennpunktvierteln vor allem mit sozialer Ausgrenzung und fehlenden Perspektiven fuer [Gruppe 1] Jugendliche. Kriminologin Mueller betonte, man duerfe keinen Generalverdacht gegen die Gruppe hegen — die grosse Mehrheit halte sich an Gesetze.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Kriminalitaet wird auf soziale Ursachen zurueckgefuehrt und eine explizite Differenzierung wird vorgenommen — das ist CRIME_FRAME_POS nach der Praventions- und Differenzierungsdefinition."
    },
    # 9. CRIME_FRAME_POS, prevention/deradicalization program
    {
        "text": "Das neue Deradikalisierungsprogramm der Bundesregierung soll vor allem [Gruppe 1] Jugendliche ansprechen, die gefaehrdet sind, in extremistische Kreise abzudriften. Sozialarbeiter begleiten die Teilnehmer ueber zwei Jahre und helfen beim Einstieg in Ausbildung und Arbeit.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Ein explizites Praeventionsprogramm im Sicherheitskontext — Schutz und Praevention stehen im Vordergrund, nicht Taeterschaft."
    },
    # 10. NO_CRIME_FRAME, integration without security frame (boundary with CRIME_FRAME_POS)
    {
        "text": "Die Stadtratsfraktion diskutierte Massnahmen zur besseren Integration von [Gruppe 1]. Themen waren Sprachkurse, Wohnraumvermittlung und Beschaeftigungsfoerderung. Ein konkreter Sicherheits- oder Kriminalitaetsbezug wurde nicht hergestellt.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Integration ohne jeden Sicherheits- oder Kriminalitaetsbezug ist NO_CRIME_FRAME — CRIME_FRAME_POS erfordert einen expliziten Sicherheits- oder Praeventionsrahmen."
    },
    # fixes id 14, 21, 135, generalised threat/disorder framing
    {
        "text": "Die Behörden berichten, dass sich in Teilen der Stadt regelrechte Parallelgesellschaften gebildet haben, in denen [Gruppe 1] unter sich bleiben und staatliche Kontrolle kaum noch greift. Anwohner klagen über Verwahrlosung und mangelnde Sicherheit.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird mit Kontrollverlust, Unordnung und mangelnder Sicherheit im deutschen Inland verknüpft — das ist generalisierte gesellschaftliche Bedrohungsrahmung und zählt als CRIME_FRAME_NEG."
    },

    # fixes id 30, explicit domestic security risk framing
    {
        "text": "Der Botschafter warnte: 'Diese [Gruppe 1] stellen ein Sicherheitsrisiko für Deutschland dar.' Es sei falsch, sie aufzunehmen, da sie in deutschen Städten für Unruhe sorgen würden.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird von einer öffentlichen Person explizit als Sicherheitsrisiko für Deutschland bezeichnet — das ist CRIME_FRAME_NEG unabhängig davon ob ein konkretes Verbrechen genannt wird."
    },
    # fixes id 15, 47, 113, 123, 189, political commentary overcalling
    {
        "text": "Die Politikerin forderte eine bessere Integrationspolitik für [Gruppe 1]. In der Debatte wurden auch Fragen der öffentlichen Ordnung angesprochen, ohne jedoch konkrete Vorfälle oder Straftaten zu nennen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Allgemeine politische Debatten über Integration oder öffentliche Ordnung ohne explizite Verknüpfung der Gruppe mit konkreten Straftaten, Verdacht oder Ermittlungen sind NO_CRIME_FRAME."
    }
]

print(f"Few-shot examples loaded: {len(FEW_SHOT_EXAMPLES)}")
for i, ex in enumerate(FEW_SHOT_EXAMPLES):
    print(f"  {i+1}. {ex['label']}")

Few-shot examples loaded: 15
  1. NO_CRIME_FRAME
  2. CRIME_FRAME_NEG
  3. CRIME_FRAME_NEG
  4. NO_CRIME_FRAME
  5. NO_CRIME_FRAME
  6. NO_CRIME_FRAME
  7. CRIME_FRAME_NEG
  8. CRIME_FRAME_NEG
  9. CRIME_FRAME_POS
  10. CRIME_FRAME_POS
  11. CRIME_FRAME_POS
  12. NO_CRIME_FRAME
  13. CRIME_FRAME_NEG
  14. CRIME_FRAME_NEG
  15. NO_CRIME_FRAME


In [13]:
def call_api(messages, temperature=0.0):
    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": messages
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    for attempt in range(3):
        try:
            r = requests.post(
                f"{HOST}/chat/completions",
                json=payload, headers=headers, timeout=120
            )
            if r.status_code == 200:
                return r.json()["choices"][0]["message"]["content"].strip()
            print(f"  [attempt {attempt+1}] status {r.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] error: {e}")
            time.sleep(5)
    return "ERROR"


def format_few_shot_block(examples):
    blocks = []
    for ex in examples:
        block = f"Text:\n{ex['text']}\n\nLabel: {ex['label']}\nExplanation: {ex['explanation']}"
        blocks.append(block)
    return "\n\n---\n\n".join(blocks)


def annotate_few_shot(text, temperature=0.0):
    few_shot_block = format_few_shot_block(FEW_SHOT_EXAMPLES)
    prompt = (
        f"{instructions}\n\n"
        f"Hier sind Beispiele fuer die Annotation:\n\n{few_shot_block}\n\n---\n\n"
        f"Jetzt annotieren Sie diesen Absatz:\n\n{text}\n\n"
        f"{reminder}\n\n"
        "Antworten Sie genau in diesem Format:\n"
        "Label: <NO_CRIME_FRAME oder CRIME_FRAME_NEG oder CRIME_FRAME_POS>\n"
        "Explanation: <ein Satz>"
    )
    messages = [
        {"role": "system", "content": "Sie sind ein Experte fuer Inhaltsanalyse. Antworten Sie immer im angegebenen Format."},
        {"role": "user",   "content": prompt}
    ]
    return call_api(messages, temperature)


print("Functions loaded.")

Functions loaded.


## Load Remaining 250 Rows

In [14]:
original  = pd.read_csv(RESULTS_DIR / "step3_testset_500_for_human_annotation.csv")
completed = pd.read_csv(RESULTS_DIR / "step3_testset_500_human_completed_translated.csv")

remaining = original[
    ~original["testset_id"].isin(completed["testset_id"])
].copy().reset_index(drop=True)

print(f"Remaining rows: {len(remaining)}")
print(f"testset_id range: {remaining['testset_id'].min()} to {remaining['testset_id'].max()}")
print(remaining["group"].value_counts().head(10))

Remaining rows: 250
testset_id range: 251 to 500
group
RUS       37
REFUG     31
UKR       25
MIGR      19
GBR       13
USA        9
MUSL       9
ASYL       8
ITA        8
ISLMST     7
Name: count, dtype: int64


## Annotate Remaining 250 with Condition B

In [15]:
OUTPUT_NEW = RESULTS_DIR / "step4_new250_model_labels.csv"

if OUTPUT_NEW.exists():
    done = pd.read_csv(OUTPUT_NEW)
    done_ids = set(done["testset_id"])
    print(f"Resuming: {len(done)} done, {len(remaining) - len(done)} remaining")
else:
    done = pd.DataFrame()
    done_ids = set()
    print(f"Starting fresh: {len(remaining)} rows")

buffer = []

for i, row in remaining.iterrows():
    if row["testset_id"] in done_ids:
        continue

    raw         = annotate_few_shot(str(row["text_block_en"]))
    label       = parse_label(raw)
    explanation = parse_explanation(raw)

    buffer.append({
        "testset_id":    row["testset_id"],
        "group":         row["group"],
        "text_block_en": row["text_block_en"],
        "model_label":   label,
        "model_explanation": explanation,
    })
    done_ids.add(row["testset_id"])

    if len(buffer) % 50 == 0:
        chunk = pd.DataFrame(buffer)
        chunk.to_csv(OUTPUT_NEW, mode="a", header=not OUTPUT_NEW.exists(), index=False)
        done  = pd.concat([done, chunk], ignore_index=True)
        buffer = []
        neg = (done["model_label"] == "CRIME_FRAME_NEG").sum()
        pos = (done["model_label"] == "CRIME_FRAME_POS").sum()
        print(f"  [{len(done)}/{len(remaining)}]  NEG: {neg}  POS: {pos}")

if buffer:
    chunk = pd.DataFrame(buffer)
    chunk.to_csv(OUTPUT_NEW, mode="a", header=not OUTPUT_NEW.exists(), index=False)
    done = pd.concat([done, chunk], ignore_index=True)

print(f"\nDone. {len(done)} rows annotated.")
print(done["model_label"].value_counts())

Resuming: 250 done, 0 remaining

Done. 250 rows annotated.
model_label
NO_CRIME_FRAME     190
CRIME_FRAME_NEG     60
Name: count, dtype: int64


## Generate Human Labelling File

Clean file for you and Robin to fill in manually.
Columns: testset_id, group, text_block_en, label_marmee, label_robin, notes.
Model labels and explanations are included so you can use them as a reference,
but fill in your own label independently first before checking the model.

In [16]:
done = pd.read_csv(OUTPUT_NEW)

# merge with original to get all original columns back
human_file = remaining.merge(
    done[["testset_id", "model_label", "model_explanation"]],
    on="testset_id",
    how="left"
)

# add empty columns for human labels
human_file["label_marmee"] = ""
human_file["label_robin"]  = ""
human_file["notes"]        = ""

# reorder, put text and human label columns first
cols = [
    "testset_id", "group",
    "text_block_en",
    "label_marmee", "label_robin", "notes",
    "model_label", "model_explanation",
]
# add any remaining original columns at the end
extra = [c for c in human_file.columns if c not in cols]
human_file = human_file[cols + extra]

output_human = RESULTS_DIR / "step4_new250_human_labelling.csv"
human_file.to_csv(output_human, index=False)

print(f"Human labelling file saved: {output_human}")
print(f"Rows: {len(human_file)}")
print(f"\nModel label distribution (for reference):")
print(human_file["model_label"].value_counts())
print(f"\nGroup distribution:")
print(human_file["group"].value_counts().head(15))

Human labelling file saved: results/step4_new250_human_labelling.csv
Rows: 250

Model label distribution (for reference):
model_label
NO_CRIME_FRAME     190
CRIME_FRAME_NEG     60
Name: count, dtype: int64

Group distribution:
group
RUS       37
REFUG     31
UKR       25
MIGR      19
GBR       13
USA        9
MUSL       9
ASYL       8
ITA        8
ISLMST     7
FRA        6
HUN        6
JUDE       5
FOR        5
JPN        4
Name: count, dtype: int64


## Evaluate Against Human Labels

Run this cell AFTER you and Robin have filled in label_marmee and label_robin.

In [17]:
labelled = pd.read_csv(RESULTS_DIR / "step4_new250_human_labelling_translated.csv")

# keep only rows where both annotators filled in a valid label
labelled = labelled[
    labelled["label_marmee"].isin(VALID_LABELS) &
    labelled["label_robin"].isin(VALID_LABELS)
].copy()

print(f"Rows with both human labels: {len(labelled)}")

# check agreement
labelled["consensus"] = labelled.apply(
    lambda r: r["label_marmee"] if r["label_marmee"] == r["label_robin"] else "UNSURE",
    axis=1
)

agreed  = labelled[labelled["consensus"] != "UNSURE"]
unsure  = labelled[labelled["consensus"] == "UNSURE"]

print(f"\nAgreed:  {len(agreed)} ({len(agreed)/len(labelled)*100:.1f}%)")
print(f"UNSURE:  {len(unsure)} ({len(unsure)/len(labelled)*100:.1f}%)")

if len(unsure) > 0:
    print(f"\nTestset IDs to review manually:")
    print(unsure["testset_id"].tolist())
    print()
    print("Details:")
    for _, row in unsure.iterrows():
        print(f"\n  id:{row['testset_id']} | group:{row['group']}")
        print(f"  Marmee: {row['label_marmee']} | Robin: {row['label_robin']}")
        print(f"  Text: {str(row['text_block_en'])[:300]}")

# human agreement metrics on agreed rows only
kappa_human = cohen_kappa_score(
    labelled["label_marmee"], labelled["label_robin"],
    labels=VALID_LABELS
)
alpha_data = np.array([
    [VALID_LABELS.index(l) for l in labelled["label_marmee"]],
    [VALID_LABELS.index(l) for l in labelled["label_robin"]]
])
alpha_human = krippendorff.alpha(alpha_data, level_of_measurement="nominal")

print(f"\nHuman-Human Agreement (all rows including UNSURE):")
print(f"  Cohen's kappa:         {kappa_human:.3f}")
print(f"  Krippendorff's alpha:  {alpha_human:.3f}")
print(f"\nConsensus label distribution:")
print(agreed["consensus"].value_counts())

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xd5 in position 13425: invalid continuation byte

In [ ]:
# merge model labels into the agreed set
model_results = pd.read_csv(RESULTS_DIR / "step4_new250_model_labels.csv")

clean = agreed.merge(
    model_results[["testset_id", "model_label", "model_explanation"]],
    on="testset_id",
    how="left"
)

y_true = clean["consensus"].tolist()
y_pred = clean["model_label"].tolist()

valid  = [(t, p) for t, p in zip(y_true, y_pred) if t in VALID_LABELS and p in VALID_LABELS]
yt, yp = zip(*valid)

acc      = accuracy_score(yt, yp)
f1_macro = f1_score(yt, yp, average="macro", labels=VALID_LABELS, zero_division=0)
f1_w     = f1_score(yt, yp, average="weighted", labels=VALID_LABELS, zero_division=0)
kappa    = cohen_kappa_score(yt, yp, labels=VALID_LABELS)
alpha_data = np.array([
    [VALID_LABELS.index(t) for t in yt],
    [VALID_LABELS.index(p) for p in yp]
])
alpha = krippendorff.alpha(alpha_data, level_of_measurement="nominal")

print(f"\n{'='*50}")
print(f"Model vs Human Consensus (fresh 250)")
print(f"{'='*50}")
print(f"Rows evaluated:        {len(yt)}")
print(f"Accuracy:              {acc:.3f}")
print(f"Macro F1:              {f1_macro:.3f}")
print(f"Weighted F1:           {f1_w:.3f}")
print(f"Cohen's kappa:         {kappa:.3f}")
print(f"Krippendorff's alpha:  {alpha:.3f}")
print()
print(classification_report(yt, yp, labels=VALID_LABELS, zero_division=0))
print("Confusion matrix (rows=human, cols=model):")
print(confusion_matrix(yt, yp, labels=VALID_LABELS))
print(f"Label order: {VALID_LABELS}")

print(f"\n=== Comparison with first 250 ===")
print(f"First 250 kappa (condition B):  0.588")
print(f"Fresh 250 kappa:                {kappa:.3f}")
if kappa >= 0.55:
    print("Result holds on fresh data.")
else:
    print("Kappa dropped, check for overfitting.")


Model vs Human Consensus (fresh 250)
Rows evaluated: 149
Accuracy:              0.792
Macro F1:              0.508
Weighted F1:           0.803
Cohen's kappa:         0.538
Krippendorff's alpha:  0.528

                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.96      0.77      0.85       112
CRIME_FRAME_NEG       0.54      0.89      0.67        36
CRIME_FRAME_POS       0.00      0.00      0.00         1

       accuracy                           0.79       149
      macro avg       0.50      0.55      0.51       149
   weighted avg       0.85      0.79      0.80       149

Confusion matrix (rows=human, cols=model):
[[86 26  0]
 [ 4 32  0]
 [ 0  1  0]]
Label order: ['NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'CRIME_FRAME_POS']

=== Comparison with Step 4 original 250 ===
Original 250 kappa:  0.642
Fresh 250 kappa:     0.538
Kappa dropped significantly — check for overfitting to original test set.
